# LLM Inference Engine — Day 1: Forward pass

**Runtime → Change runtime type → T4 GPU** before running anything.

Goal today: our own Qwen2.5-0.5B forward pass that matches Hugging Face's logits within 1e-2 and produces identical greedy text.

Flow:
1. Setup + clone your repo
2. Fill the two TODOs in `engine/model.py` (RMSNorm, MLP)
3. Run the offline architecture test (no download, instant)
4. Run the real test against Qwen2.5-0.5B
5. Generate some text with your engine
6. Commit + push

## 1. Setup

In [ ]:
!nvidia-smi -L
!pip install -q transformers safetensors accelerate

In [ ]:
# Clone your repo. Create it on GitHub first (empty, named llm-engine), push the scaffold, then:
REPO = "https://github.com/ArmaanAgarwal/llm-engine.git"
!git clone $REPO
%cd llm-engine
!ls -R engine tests

## 2. Fill the TODOs

Open `engine/model.py` in the Colab file browser (left sidebar → folder icon → `llm-engine/engine/model.py`). Two methods raise `NotImplementedError`:

**`RMSNorm.forward`** — three lines. Cast to float32, divide by root-mean-square, cast back, multiply by weight. The comment above it has the exact steps.

**`MLP.forward`** — one line. `down(silu(gate(x)) * up(x))`.

Everything else (RoPE, attention, GQA, weight loading) is written and verified. Read it; the comments explain each piece. Save the file when done.

## 3. Offline architecture test

Builds a tiny random Qwen2 inside `transformers`, copies its weights into **your** model, compares logits. No download. If your two TODOs are right, this prints `PASS` with diff 0.00000.

In [ ]:
!python -m tests.test_logits --tiny

## 4. Real test: Qwen2.5-0.5B

Downloads ~1 GB once. Loads HF's model and yours, compares logits on 5 prompts, then greedy-decodes 30 tokens with both. Must print `PASS`.

Tolerance is 1e-2 because the two implementations order their float ops slightly differently; a diff in the 1e-3 range is normal. A diff over 0.1 means a weight is mapped wrong.

In [ ]:
!python -m tests.test_logits

## 5. Generate with your engine

In [ ]:
import torch, time
from transformers import AutoTokenizer
from engine.model import from_pretrained

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = from_pretrained(device="cuda", dtype=torch.float32)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

msgs = [{"role": "user", "content": "Explain why GPUs are good at matrix multiplication in two sentences."}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")

torch.cuda.synchronize(); t0 = time.time()
out = model.generate(ids, max_new_tokens=64)
torch.cuda.synchronize(); dt = time.time() - t0

print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
print(f"\n{64/dt:.1f} tok/s  (Day 1 baseline: no cache, fp32. This is the number we'll crush tomorrow.)")

## 6. Commit

Write down the tok/s number above; it's the "before" for the whole project.

In [ ]:
!git config user.email "agarwal.armaan27@gmail.com" && git config user.name "Armaan Agarwal"
!git add -A && git commit -m "Day 1: forward pass matches HF logits" && git push

## Before you close: can you answer these without looking?

1. What does RMSNorm do and why compute it in float32?
2. Why does the model need RoPE at all?
3. 14 query heads, 2 KV heads: what does that buy us?
4. Why is Day 1 generation slow? (Hint: what does `generate` recompute every step?)

If any is fuzzy, ask before Day 2. #4 is the KV cache; that's tomorrow.